In [21]:
# Import libraries
import pandas as pd
import sqlite3
import numpy as np
import sqlalchemy
import psycopg2
import matplotlib
import seaborn

In [22]:
# Load the Netflix dataset
df = pd.read_csv("netflix.csv")

In [23]:
# Display first 5 rows
df.head()

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"


In [24]:
df.shape

(8790, 10)

In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8790 entries, 0 to 8789
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8790 non-null   object
 1   type          8790 non-null   object
 2   title         8790 non-null   object
 3   director      8790 non-null   object
 4   country       8790 non-null   object
 5   date_added    8790 non-null   object
 6   release_year  8790 non-null   int64 
 7   rating        8790 non-null   object
 8   duration      8790 non-null   object
 9   listed_in     8790 non-null   object
dtypes: int64(1), object(9)
memory usage: 686.8+ KB


In [26]:
df.describe()

,release_year
count,8790.000000
mean,2014.183163
std,8.825466
min,1925.000000
25%,2013.000000
50%,2017.000000
75%,2019.000000
max,2021.000000


In [27]:
df.isnull().sum()

show_id         0
type            0
title           0
director        0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
dtype: int64

In [28]:
# Replace missing values

df['director'] = df['director'].fillna('Unknown')
df['country'] = df['country'].fillna('Unknown')
df['rating'] = df['rating'].fillna('Not Rated')

# Convert date column to datetime
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')

In [29]:
# Remove duplicates
df = df.drop_duplicates()

# Check new shape
df.shape

(8790, 10)

In [30]:
df['type'] = df['type'].str.strip()
df['title'] = df['title'].str.strip()
df['country'] = df['country'].str.strip()
df['director'] = df['director'].str.strip()

In [31]:
# Extract numeric duration
df['duration_value'] = df['duration'].str.extract(r'(\d+)')

# Extract duration type
df['duration_type'] = df['duration'].str.extract(r'([A-Za-z]+)')

In [32]:
# Extract year when content was added
df['year_added'] = df['date_added'].dt.year

# Extract month when content was added
df['month_added'] = df['date_added'].dt.month

# View updated dataset
df.head()

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in,duration_value,duration_type,year_added,month_added
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,2021-09-25,2020,PG-13,90 min,Documentaries,90,min,2021,9
1,s3,TV Show,Ganglands,Julien Leclercq,France,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",1,Season,2021,9
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,2021-09-24,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries",1,Season,2021,9
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,2021-09-22,2021,TV-PG,91 min,"Children & Family Movies, Comedies",91,min,2021,9
4,s8,Movie,Sankofa,Haile Gerima,United States,2021-09-24,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies",125,min,2021,9


In [33]:
df[['date_added','year_added','month_added']].head()

,date_added,year_added,month_added
0,2021-09-25,2021,9
1,2021-09-24,2021,9
2,2021-09-24,2021,9
3,2021-09-22,2021,9
4,2021-09-24,2021,9


In [34]:
conn = sqlite3.connect("netflix_db.sqlite")

print("Database created successfully")

Database created successfully


In [35]:
df.to_sql(
    name="netflix_cleaned",
    con=conn,
    if_exists="replace",
    index=False
)

print("Table created successfully")

Table created successfully


In [36]:
query = "SELECT * FROM netflix_cleaned LIMIT 10"
pd.read_sql(query, conn)

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in,duration_value,duration_type,year_added,month_added
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,2021-09-25 00:00:00,2020,PG-13,90 min,Documentaries,90,min,2021,9
1,s3,TV Show,Ganglands,Julien Leclercq,France,2021-09-24 00:00:00,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",1,Season,2021,9
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,2021-09-24 00:00:00,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries",1,Season,2021,9
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,2021-09-22 00:00:00,2021,TV-PG,91 min,"Children & Family Movies, Comedies",91,min,2021,9
4,s8,Movie,Sankofa,Haile Gerima,United States,2021-09-24 00:00:00,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies",125,min,2021,9
5,s9,TV Show,The Great British Baking Show,Andy Devonshire,United Kingdom,2021-09-24 00:00:00,2021,TV-14,9 Seasons,"British TV Shows, Reality TV",9,Seasons,2021,9
6,s10,Movie,The Starling,Theodore Melfi,United States,2021-09-24 00:00:00,2021,PG-13,104 min,"Comedies, Dramas",104,min,2021,9
7,s939,Movie,Motu Patlu in the Game of Zones,Suhas Kadav,India,2021-05-01 00:00:00,2019,TV-Y7,87 min,"Children & Family Movies, Comedies, Music & Mu...",87,min,2021,5
8,s13,Movie,Je Suis Karl,Christian Schwochow,Germany,2021-09-23 00:00:00,2021,TV-MA,127 min,"Dramas, International Movies",127,min,2021,9
9,s940,Movie,Motu Patlu in Wonderland,Suhas Kadav,India,2021-05-01 00:00:00,2013,TV-Y7,76 min,"Children & Family Movies, Music & Musicals",76,min,2021,5


In [37]:
df.to_csv("netflix_cleaned.csv", index=False)

print("Cleaned dataset saved successfully")

Cleaned dataset saved successfully


In [38]:
print(df.columns)

Index(['show_id', 'type', 'title', 'director', 'country', 'date_added',
       'release_year', 'rating', 'duration', 'listed_in', 'duration_value',
       'duration_type', 'year_added', 'month_added'],
      dtype='object')


In [39]:
df = pd.read_csv("netflix_cleaned.csv")
print(df.columns)

Index(['show_id', 'type', 'title', 'director', 'country', 'date_added',
       'release_year', 'rating', 'duration', 'listed_in', 'duration_value',
       'duration_type', 'year_added', 'month_added'],
      dtype='object')
